# Agulhas Region — GDP Drifter Exploration

**Goal:** Load NOAA GDP drifter data via CloudDrift, filter to the Agulhas region, and inspect coverage for building a surface-transport transition matrix.

**Agulhas bounding box:** `lon: 10°E – 40°E`, `lat: 45°S – 25°S`

**Physics note:** Undrogued drifters (drogue lost) track surface + wind-driven transport — closest proxy to oil spill drift.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from clouddrift.datasets import gdp_6h
from clouddrift import ragged

## 1. Load GDP 6-hourly dataset
Streams from cloud — first load takes a minute, subsequent loads are cached.

In [ ]:
ds = gdp_6h()
print(ds)
print(f"\nTotal trajectories : {ds.dims['traj']}")
print(f"Total observations : {ds.dims['obs']}")

## 2. Filter to Agulhas bounding box
Keep only observations within `lon: 10–40°E`, `lat: 45–25°S`.

In [ ]:
LON_MIN, LON_MAX = 10.0, 40.0
LAT_MIN, LAT_MAX = -45.0, -25.0

lon = ds["lon"].values
lat = ds["lat"].values

mask = (
    (lon >= LON_MIN) & (lon <= LON_MAX) &
    (lat >= LAT_MIN) & (lat <= LAT_MAX)
)

print(f"Observations in Agulhas box : {mask.sum():,}")
print(f"Fraction of total           : {mask.mean():.2%}")

## 3. Identify trajectories that pass through the region

In [ ]:
rowsize = ds["rowsize"].values  # number of obs per trajectory
traj_ids = ds["id"].values

# assign trajectory index to each observation
traj_idx = np.repeat(np.arange(len(rowsize)), rowsize)

# trajectories with at least one obs in the box
trajs_in_box = np.unique(traj_idx[mask])
print(f"Trajectories passing through Agulhas box: {len(trajs_in_box):,}")

## 4. Split drogued vs undrogued
Undrogued = surface + windage ≈ oil spill proxy.

In [ ]:
drogue = ds["drogue_status"].values  # 1 = drogued, 0 = undrogued

mask_undrogued = mask & (drogue == 0)
mask_drogued   = mask & (drogue == 1)

print(f"Undrogued obs in box (oil proxy) : {mask_undrogued.sum():,}")
print(f"Drogued obs in box (15m current) : {mask_drogued.sum():,}")

## 5. Map — all drifter tracks in Agulhas box

In [ ]:
fig, ax = plt.subplots(
    figsize=(10, 8),
    subplot_kw={"projection": ccrs.PlateCarree()}
)

ax.set_extent([LON_MIN - 2, LON_MAX + 2, LAT_MIN - 2, LAT_MAX + 2], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, color="lightgray", zorder=2)
ax.add_feature(cfeature.COASTLINE, linewidth=0.5, zorder=3)
ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.5)

ax.scatter(
    lon[mask_drogued], lat[mask_drogued],
    s=0.3, c="steelblue", alpha=0.4, label="Drogued (15m)", zorder=4
)
ax.scatter(
    lon[mask_undrogued], lat[mask_undrogued],
    s=0.3, c="darkorange", alpha=0.5, label="Undrogued (surface/oil proxy)", zorder=5
)

ax.set_title("GDP Drifter Observations — Agulhas Region", fontsize=13)
ax.legend(markerscale=10, loc="lower right")
plt.tight_layout()
plt.savefig("../figures/agulhas_gdp_coverage.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Temporal coverage

In [ ]:
import pandas as pd

time_in_box = ds["time"].values[mask]
years = pd.DatetimeIndex(time_in_box).year

fig, ax = plt.subplots(figsize=(10, 3))
ax.hist(years, bins=np.arange(years.min(), years.max() + 2) - 0.5, color="steelblue", edgecolor="white")
ax.set_xlabel("Year")
ax.set_ylabel("Observations")
ax.set_title("Temporal Distribution of GDP Drifter Obs in Agulhas Box")
plt.tight_layout()
plt.show()

## Next steps
- If coverage is sufficient → build transition matrix from undrogued tracks
- If coverage is sparse → supplement with CMEMS + ERA5 particle simulation
- Add mass-decay model: `m(t) = m0 * exp(-λt)` where λ ≈ 1/14 days (oil half-life ~2 weeks)